# Unit 3 Assignment: Building a Production Advanced RAG System

This notebook builds the full advanced RAG pipeline:
1. Document corpus setup
2. Hybrid retrieval with BM25 + SBERT + Reciprocal Rank Fusion (RRF)
3. Cross-encoder re-ranking
4. Query expansion with Multi-Query
5. End-to-end advanced RAG pipeline
6. Naive vs advanced comparison experiment

In [ ]:
# Use known compatible package versions for this notebook.
from importlib.util import find_spec
import subprocess
import sys

required_packages = {
    'rank_bm25': 'rank-bm25',
    'sentence_transformers': 'sentence-transformers==2.7.0',
    'transformers': 'transformers==4.41.2',
    'accelerate': 'accelerate>=0.30.0',
    'google.generativeai': 'google-generativeai',
    'pandas': 'pandas',
}

for module_name, package_name in required_packages.items():
    if find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])

# Force compatible versions even if a broken newer install already exists.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sentence-transformers==2.7.0', 'transformers==4.41.2', 'accelerate>=0.30.0'])

print('Dependencies are ready.')

In [ ]:
import os
import re
from typing import Dict, List

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from sentence_transformers.util import cos_sim

try:
    import google.generativeai as genai
except ImportError:
    genai = None

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')

def get_gemini_model(model_name: str = 'gemini-1.5-flash'):
    if not GEMINI_API_KEY or genai is None:
        return None
    genai.configure(api_key=GEMINI_API_KEY)
    return genai.GenerativeModel(model_name)

def simple_tokenize(text: str) -> List[str]:
    return re.findall(r'[a-z0-9]+', text.lower())

class SimpleVectorizer:
    def __init__(self, corpus: List[str]):
        vocab = sorted({token for doc in corpus for token in simple_tokenize(doc)})
        self.vocab = {token: idx for idx, token in enumerate(vocab)}

    def encode(self, texts):
        single_input = isinstance(texts, str)
        if single_input:
            texts = [texts]

        matrix = np.zeros((len(texts), len(self.vocab)), dtype=float)
        for row_idx, text in enumerate(texts):
            for token in simple_tokenize(text):
                if token in self.vocab:
                    matrix[row_idx, self.vocab[token]] += 1.0

        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        matrix = matrix / norms
        return matrix[0] if single_input else matrix

def lexical_similarity(query: str, document: str) -> float:
    query_tokens = set(simple_tokenize(query))
    doc_tokens = set(simple_tokenize(document))
    if not query_tokens or not doc_tokens:
        return 0.0
    overlap = len(query_tokens & doc_tokens)
    return overlap / ((len(query_tokens) * len(doc_tokens)) ** 0.5)

print('Gemini configured:', bool(GEMINI_API_KEY and genai is not None))

## Part 1 - Document Corpus Setup

In [ ]:
corpus = [
    'When people say Transformers capture meaning, they are usually talking about self-attention. Each word can look at the other words in the sentence, so its representation changes depending on the context around it.',
    'Inside a Transformer encoder, the same sentence passes through repeated attention and feed-forward blocks. Positional encodings are added because the model still needs some signal about word order.',
    'The paper Attention Is All You Need made Transformers popular by showing that pure attention could replace recurrent layers for sequence tasks like machine translation.',
    'Gradient descent is the basic training loop: compute the loss, backpropagate, and nudge the weights in the direction that lowers error. In practice, we usually do this with mini-batches instead of the full dataset.',
    'Adam is often the default optimizer in deep learning projects because it mixes momentum with adaptive learning rates. It usually gets to a reasonable solution faster than plain SGD.',
    'Learning-rate schedules matter more than they seem at first. Warmup can prevent unstable early updates, while cosine decay or step decay helps training settle down later.',
    'Batch normalization made training deep networks much easier in practice. By keeping activations in a healthier range, it often lets the model train faster and with less sensitivity to initialization.',
    'Dropout is a simple trick but still useful: during training, some neurons are randomly turned off so the network does not become too dependent on a small set of features.',
    'The vanishing gradient problem shows up when early layers barely receive a learning signal. Residual connections and ReLU activations were important partly because they made this issue less painful.',
    'Embeddings turn words or tokens into vectors, which gives the model a way to place similar meanings close to each other. That is why terms like king, queen, and prince often end up related in embedding space.',
    'LSTMs were designed to handle longer dependencies than vanilla RNNs. Their input, forget, and output gates give them a better shot at deciding what information to keep or throw away.',
    'LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning method where you train small low-rank update matrices instead of editing every weight in a large language model.',
]

print(f'Corpus size: {len(corpus)} documents')
print('Transformer-related docs: 3')
print('Optimization-related docs: 4')
print('Keyword-heavy proper noun document included: LoRA')

## Part 2 - Hybrid Retrieval

In [ ]:
class HybridRetriever:
    """Combine keyword search and dense retrieval, then merge the ranks with RRF."""

    def __init__(self, corpus: List[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        self.tokenized_corpus = [simple_tokenize(doc) for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        self.semantic_backend = 'sbert'
        self.sbert = None
        self.vectorizer = None

        try:
            self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
            self.corpus_embeddings = self.sbert.encode(
                corpus,
                convert_to_tensor=True,
                normalize_embeddings=True,
            )
        except Exception:
            self.semantic_backend = 'simple'
            self.vectorizer = SimpleVectorizer(corpus)
            self.corpus_embeddings = self.vectorizer.encode(corpus)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        bm25_scores = self.bm25.get_scores(simple_tokenize(query))
        bm25_order = np.argsort(-bm25_scores)
        bm25_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(bm25_order)}

        if self.semantic_backend == 'sbert':
            query_embedding = self.sbert.encode(
                query,
                convert_to_tensor=True,
                normalize_embeddings=True,
            )
            sbert_scores = cos_sim(query_embedding, self.corpus_embeddings)[0].cpu().numpy()
        else:
            query_embedding = self.vectorizer.encode(query)
            sbert_scores = self.corpus_embeddings @ query_embedding

        sbert_order = np.argsort(-sbert_scores)
        sbert_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(sbert_order)}

        fused = []
        for doc_id, text in enumerate(self.corpus):
            bm25_rank = bm25_ranks[doc_id]
            sbert_rank = sbert_ranks[doc_id]
            rrf_score = (1 / (self.k + bm25_rank)) + (1 / (self.k + sbert_rank))
            fused.append({
                'doc_id': doc_id,
                'rrf_score': float(rrf_score),
                'bm25_rank': bm25_rank,
                'sbert_rank': sbert_rank,
                'text': text,
            })

        fused.sort(key=lambda item: item['rrf_score'], reverse=True)
        return fused[:top_k]


hybrid_retriever = HybridRetriever(corpus, k=60)
print('Semantic backend:', hybrid_retriever.semantic_backend)
hybrid_retriever.retrieve('how do transformers encode meaning?', top_k=3)

## Part 3 - Cross-Encoder Re-Ranker

In [ ]:
try:
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    reranker_backend = 'cross-encoder'
except Exception:
    cross_encoder = None
    reranker_backend = 'token-overlap'

print('Reranker backend:', reranker_backend)

def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    """Re-rank candidates using the cross-encoder when available."""
    if cross_encoder is not None:
        pairs = [(query, candidate['text']) for candidate in candidates]
        scores = cross_encoder.predict(pairs)
    else:
        scores = [lexical_similarity(query, candidate['text']) for candidate in candidates]

    reranked = []
    for candidate, score in zip(candidates, scores):
        updated = candidate.copy()
        updated['cross_encoder_score'] = float(score)
        reranked.append(updated)

    reranked.sort(key=lambda item: item['cross_encoder_score'], reverse=True)
    return reranked[:top_k]


sample_candidates = hybrid_retriever.retrieve('optimization techniques for training', top_k=5)
rerank('optimization techniques for training', sample_candidates, top_k=3)

## Part 4 - Query Expansion (Multi-Query)

In [ ]:
class MultiQueryExpander:
    """Generate alternate phrasings for the same question."""

    def __init__(self, model_name: str = 'gemini-1.5-flash'):
        self.model = get_gemini_model(model_name)

    def _fallback_queries(self, query: str) -> List[str]:
        """Fallback when Gemini API is not available."""
        cleaned = query.strip().rstrip('?')
        return [
            cleaned,
            f'Explain the core concept behind: {cleaned}',
            f'Give the machine learning terminology related to: {cleaned}',
        ]

    def expand(self, query: str) -> List[str]:
        if self.model is None:
            print('(Using fallback queries - no Gemini API key)')
            return self._fallback_queries(query)

        prompt = f'''Generate exactly 3 short search-query paraphrases for this student question.
Keep them technical, distinct, and useful for document retrieval.
Return one query per line with no numbering.

Question: {query}
'''
        try:
            response = self.model.generate_content(
                prompt,
                generation_config={'temperature': 0.0},
            )
            lines = [line.strip('- ').strip() for line in response.text.splitlines() if line.strip()]
            deduped = []
            for line in lines:
                if line not in deduped:
                    deduped.append(line)
            return deduped[:3] if len(deduped) >= 3 else self._fallback_queries(query)
        except Exception as e:
            print(f'Gemini error: {e}')
            return self._fallback_queries(query)


expander = MultiQueryExpander()
expanded = expander.expand('how do transformers encode meaning?')
print('Expanded queries:')
for i, q in enumerate(expanded, 1):
    print(f'  {i}. {q}')

## Part 5 - End-to-End Pipeline

In [ ]:
class NaiveDenseRetriever:
    """Dense-only retriever for comparison."""
    
    def __init__(self, corpus: List[str], hybrid_backend: HybridRetriever):
        self.corpus = corpus
        self.semantic_backend = hybrid_backend.semantic_backend
        self.sbert = hybrid_backend.sbert
        self.vectorizer = hybrid_backend.vectorizer
        self.corpus_embeddings = hybrid_backend.corpus_embeddings

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        if self.semantic_backend == 'sbert':
            query_embedding = self.sbert.encode(
                query,
                convert_to_tensor=True,
                normalize_embeddings=True,
            )
            scores = cos_sim(query_embedding, self.corpus_embeddings)[0].cpu().numpy()
        else:
            query_embedding = self.vectorizer.encode(query)
            scores = self.corpus_embeddings @ query_embedding
        order = np.argsort(-scores)[:top_k]
        return [
            {
                'doc_id': int(doc_id),
                'score': float(scores[doc_id]),
                'text': self.corpus[int(doc_id)],
            }
            for doc_id in order
        ]


naive_retriever = NaiveDenseRetriever(corpus, hybrid_retriever)

def expand_and_retrieve(user_query: str, top_k_per_query: int = 4) -> List[Dict]:
    """Expand query and retrieve from all variants, deduplicating by text."""
    expanded_queries = expander.expand(user_query)
    pooled = {}

    for query_variant in expanded_queries:
        retrieved = hybrid_retriever.retrieve(query_variant, top_k=top_k_per_query)
        for item in retrieved:
            text_key = item['text']
            if text_key not in pooled or item['rrf_score'] > pooled[text_key]['rrf_score']:
                pooled[text_key] = item

    return list(pooled.values())

def build_context(reranked_docs: List[Dict]) -> str:
    return '\n\n'.join(
        f"Document {idx}: {doc['text']}"
        for idx, doc in enumerate(reranked_docs, start=1)
    )

def generate_answer(user_query: str, reranked_docs: List[Dict]) -> str:
    context = build_context(reranked_docs)
    model = get_gemini_model('gemini-1.5-flash')

    if model is not None:
        prompt = f'''Answer the student's question using only the evidence below.
If the context is limited, say so briefly.

Question: {user_query}

Context:
{context}
'''
        try:
            response = model.generate_content(
                prompt,
                generation_config={'temperature': 0.2},
            )
            return response.text.strip()
        except Exception:
            pass

    # Fallback answer without API
    summary_lines = [f'- {doc["text"]}' for doc in reranked_docs]
    return (
        f"Question: {user_query}\n\n"
        "Relevant notes from the retrieved documents:\n"
        + '\n'.join(summary_lines)
        + '\n\nBased on the reranked passages above, the first document should be treated as the strongest evidence.'
    )

def advanced_rag_trace(user_query: str) -> Dict:
    """Full pipeline with trace info for debugging."""
    expanded_candidates = expand_and_retrieve(user_query, top_k_per_query=4)
    reranked = rerank(user_query, expanded_candidates, top_k=3)
    answer = generate_answer(user_query, reranked)
    return {
        'query': user_query,
        'expanded_queries': expander.expand(user_query),
        'candidates': expanded_candidates,
        'reranked': reranked,
        'answer': answer,
    }

def advanced_rag(user_query: str) -> str:
    """Full pipeline: Query Expansion -> Hybrid Retrieval -> Re-Ranking -> LLM Generation"""
    return advanced_rag_trace(user_query)['answer']


trace = advanced_rag_trace('how do transformers encode meaning?')
print('Expanded queries:', trace['expanded_queries'])
print('\nTop reranked document:')
print(trace['reranked'][0]['text'])
print('\nGenerated answer:')
print(trace['answer'])

## Part 6 - Comparison Experiment

In [ ]:
test_queries = [
    'how do transformers encode meaning?',
    'optimization techniques for training',
    'what is LoRA used for in language models?',
]

rows = []
for query in test_queries:
    naive_top = naive_retriever.retrieve(query, top_k=1)[0]['text']
    advanced_top = advanced_rag_trace(query)['reranked'][0]['text']
    rows.append({
        'Query': query,
        'Naive RAG Top Doc': naive_top[:100] + '...',
        'Advanced RAG Top Doc': advanced_top[:100] + '...',
        'Different?': 'Yes' if naive_top != advanced_top else 'No',
    })

comparison_df = pd.DataFrame(rows)
print('Comparison Results:')
comparison_df

## Comparison Table

| Query | Naive RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| `how do transformers encode meaning?` | (Run cells to see) | (Run cells to see) | (Run cells to see) |
| `optimization techniques for training` | (Run cells to see) | (Run cells to see) | (Run cells to see) |
| `what is LoRA used for in language models?` | (Run cells to see) | (Run cells to see) | (Run cells to see) |

**Observation:** The advanced pipeline provides better traceability with BM25 rank, SBERT rank, RRF score, and cross-encoder scores, even when results are similar.